In [ ]:
!pip install streamlit pyngrok psutil scikit-learn joblib

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 300

data = pd.DataFrame({
    "study_time": np.random.uniform(1,6,n),
    "break_time": np.random.uniform(0.2,2,n),
    "sleep_hours": np.random.uniform(3,9,n),
    "screen_time": np.random.uniform(2,10,n),
    "typing_rate": np.random.uniform(20,80,n),
    "cpu_usage": np.random.uniform(10,90,n)
})

data["focus_ratio"] = data["study_time"]/(data["study_time"]+data["break_time"])
data["sleep_debt"] = np.maximum(0,8-data["sleep_hours"])
data["fatigue_index"] = data["screen_time"]*data["sleep_debt"]

data["label"] = (
    (data["sleep_debt"]>2).astype(int) +
    (data["screen_time"]>6).astype(int)
)

data.to_csv("data.csv",index=False)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import joblib

df = pd.read_csv("data.csv")

X = df.drop("label",axis=1)
y = df["label"]

model = RandomForestClassifier(n_estimators=100)
model.fit(X,y)

joblib.dump(model,"burnout_model.pkl")


['burnout_model.pkl']

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import psutil

st.title("🧠 CognitiveLoad AI Prototype")

model = joblib.load("burnout_model.pkl")

study_time = st.slider("Study Time (hrs)",1.0,8.0,3.0)
break_time = st.slider("Break Time (hrs)",0.1,3.0,1.0)
sleep_hours = st.slider("Sleep Hours",1.0,10.0,6.0)
screen_time = st.slider("Screen Time",1.0,12.0,5.0)
typing_rate = st.slider("Typing Rate",10,100,40)

cpu_usage = psutil.cpu_percent()

focus_ratio = study_time/(study_time+break_time)
sleep_debt = max(0,8-sleep_hours)
fatigue_index = screen_time*sleep_debt

input_df = pd.DataFrame([{
    "study_time":study_time,
    "break_time":break_time,
    "sleep_hours":sleep_hours,
    "screen_time":screen_time,
    "typing_rate":typing_rate,
    "cpu_usage":cpu_usage,
    "focus_ratio":focus_ratio,
    "sleep_debt":sleep_debt,
    "fatigue_index":fatigue_index
}])

prediction = model.predict_proba(input_df)[0]
burnout_score = int(prediction[1]*100)

st.metric("🔥 Burnout Risk Score",burnout_score)

if burnout_score>60:
    st.warning("High fatigue risk detected.")
else:
    st.success("Focus level stable.")


Overwriting app.py


In [ ]:
from pyngrok import ngrok

# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# and replace 'YOUR_AUTH_TOKEN' with it.
ngrok.set_auth_token('39ps0AveeZHqReesuGo5wb2IxCy_61XGS17Tbd5c22QYyvZDi')

!streamlit run app.py &>/dev/null&
public_url = ngrok.connect(8501)
public_url

<NgrokTunnel: "https://frightfully-unliveried-gavin.ngrok-free.dev" -> "http://localhost:8501">